In [1]:
import re
import os
import time
import spacy
import string
import regex
import pickle
import sqlite3
import inflect
import treelib
import threading
import numpy as np
import pandas as pd
from tqdm import tqdm
from os import listdir
from treelib import Tree
from pandas import DataFrame
from taxonerd import TaxoNERD
from os.path import isfile, join
from IPython.display import clear_output
from rich.console import Console
from itertools import islice
from spacy.util import filter_spans
from spacy.tokens import Token, Doc, Span
from typing import List, Dict, Tuple, Set, Optional, Union, Literal
# !pip install duckdb
# import duckdb

In [337]:
print(df[df["col:species"] == "Panthera leo"])

        col:ID col:status         col:scientificName    col:rank  \
5178570  5K5L8   accepted           Panthera leo leo  subspecies   
5241285  7KGW9   accepted  Panthera leo melanochaita  subspecies   
5981691  F9JJ3   accepted               BOLD:AAD6819    unranked   
5981692  F9JJ4   accepted               BOLD:AEW2394    unranked   

        col:originalSpelling col:uninomial col:genericName  \
5178570                 None          None        Panthera   
5241285                 None          None        Panthera   
5981691                 None          None            None   
5981692                 None          None            None   

        col:infragenericEpithet col:specificEpithet col:infraspecificEpithet  \
5178570                    None                 leo                      leo   
5241285                    None                 leo             melanochaita   
5981691                    None                None                     None   
5981692                    N

In [8]:
conn = sqlite3.connect("./db/COL.db")
cursor = conn.cursor()

In [ ]:
# conn = duckdb.connect("./db/COL.db")
# cursor = conn.cursor()

In [326]:
vern = pd.read_sql("""
    SELECT *
    FROM VernacularName
    WHERE "col:language" = 'eng'
""", conn)

In [330]:
joined = pd.read_sql("""
    SELECT *
    FROM JoinedTable
""", conn)

In [327]:
print(vern.columns)
vern["col:taxonID"].is_unique
vern[vern["col:taxonID"].duplicated()]

Index(['col:taxonID', 'col:name', 'col:language'], dtype='object')


,col:taxonID,col:name,col:language
8,6SYS3,lobate fig sponge,eng
13,723P6,urn calcareous sponge,eng
17,3H87X,purse sponge,eng
21,6PSD5,bristly vase calcareous sponge,eng
29,6PTPF,orange pipe sponge,eng
...,...,...,...
214978,mNIprzdlDYtf8WqPtzykB1,Giant South American River Turtle,eng
214979,mNIprzdlDYtf8WqPtzykB1,True Amazon Turtle,eng
214981,1YbFAWXMr0N--OSW0EllI,Yellow-spotted Sideneck Turtle,eng
214983,ciwGcxzgIic_8WcbuQNzV0,Red-headed River Turtle,eng


In [328]:
print(df.columns)
df["col:ID"].is_unique
df[df["col:ID"].duplicated()]

Index(['col:ID', 'col:status', 'col:scientificName', 'col:rank',
       'col:originalSpelling', 'col:uninomial', 'col:genericName',
       'col:infragenericEpithet', 'col:specificEpithet',
       'col:infraspecificEpithet', 'col:code', 'col:nameStatus',
       'col:environment', 'col:species', 'col:subgenus', 'col:genus',
       'col:subfamily', 'col:family', 'col:superfamily', 'col:suborder',
       'col:order', 'col:subclass', 'col:class', 'col:subphylum', 'col:phylum',
       'col:kingdom'],
      dtype='object')


,col:ID,col:status,col:scientificName,col:rank,col:originalSpelling,col:uninomial,col:genericName,col:infragenericEpithet,col:specificEpithet,col:infraspecificEpithet,...,col:subfamily,col:family,col:superfamily,col:suborder,col:order,col:subclass,col:class,col:subphylum,col:phylum,col:kingdom


In [332]:
# joined[joined["VernacularName"].str.contains('Giant South American River Turtl')]
joined[joined["ScientificName"].str.contains('Podocnemis expansa')]

,ID,ScientificName,Genus,Rank,VernacularName,GenericName,SpecificEpithet
163362,4KQG8,Podocnemis expansa,Podocnemis,species,Arrau,Podocnemis,expansa
163363,4KQG8,Podocnemis expansa,Podocnemis,species,Giant Amazon River Turtle,Podocnemis,expansa
163364,4KQG8,Podocnemis expansa,Podocnemis,species,Giant South American River Turtle,Podocnemis,expansa
163365,4KQG8,Podocnemis expansa,Podocnemis,species,True Amazon Turtle,Podocnemis,expansa
203492,4KQG8,Podocnemis expansa,Podocnemis,species,Arrau River Turtle,Podocnemis,expansa
203493,4KQG8,Podocnemis expansa,Podocnemis,species,South American River Turtle,Podocnemis,expansa


In [333]:
len(joined)

204595

In [ ]:
df = pd.read_sql("""
    SELECT *
    FROM NameUsage
""", conn)

In [ ]:
df.columns

In [301]:
# species = set(df["col:genus"].tolist())
scientific_names = set(df["col:scientificName"].tolist())

print(len(species))
print(len(scientific_names))
print(len(species & scientific_names))

[*scientific_names - species][:50]

229791
7648104
229790


['Placusa nairobiana',
 'Aiouea amplexicaulis',
 'Zygaena vogesiaca',
 'BOLD:ACN7130',
 'SH1160565.10FU',
 'Isolona pleurocarpa',
 'Tipula (Triplicitipula) doaneiana',
 'Fusicoccum mangiferae',
 'SH0118222.10FU',
 'Cratolampis',
 'SH0131906.10FU',
 'SH0804569.10FU',
 'Pachysmopoda abbreviata',
 'Huperzia cleefiana',
 'Lycium lineare',
 'Signiphora falcata',
 'Orthops alpicola',
 'Syntomus antoinei',
 'Prochaetosoma arcticum',
 'Cytheropteron branchium',
 'Asarum caulescens',
 'Chrysopa kiangsuensis',
 'SH1378344.10FU',
 'Pycnostachys graminifolia',
 'Ophiosphalma serratum',
 'SH0159551.10FU',
 'Diaporthe chamaeropicola',
 'Lethe ocellata lyncus',
 'Turritella australis',
 'Philarctus bergrothi',
 'Leptonia modesta',
 'Ostedes (Ostedes) dentata',
 'Bioculus belvederi',
 'SH1368257.10FU',
 'Euryphorus hemipterus',
 'BOLD:ADI2737',
 'Polystichum caripense',
 'SH0152560.10FU',
 'BOLD:AAJ2247',
 'BOLD:AET0417',
 'SH0939847.10FU',
 'Spongostylum variabile',
 'BOLD:AAD6447',
 'Cyrtochloa majo

In [305]:
'Panthera urus' in scientific_names

False

In [364]:
cols = [
    ["Kingdom", "col:kingdom"],
    ["Phylum", "col:phylum"],
    ["SubPhylum", "col:subphylum"],
    ["Class", "col:class"],
    ["SubClass", "col:subclass"],
    ["Order", "col:order"],
    ["SubOrder", "col:suborder"],
    ["SuperFamily", "col:superfamily"],
    ["Family", "col:family"],
    ["SubFamily", "col:subfamily"],
    ["Genus", "col:genus"],
    ["SubGenus", "col:subgenus"],
    ["GenericName", "col:genericName"],
    ["SpecificEpithet", "col:specificEpithet"],
    ["IntraspecificEpithet", "col:infraspecificEpithet"],
    ["Species", "col:species"],
    ["ScientificName", "col:scientificName"],
]

In [ ]:
select = ", ".join([f'"{col[1]}"' for col in cols])
print(f"Select: {select}")

In [ ]:
# Didn't Work Out
# Too much rows, I don't think I can speed it up enough.
# def create_levels(cursor):
#     s = []
#     t = Tree()
#     t.create_node(identifier="*")

#     # Initialize Stack
#     kingdoms = {row[0] for row in cursor.execute('''SELECT DISTINCT "col:kingdom" FROM NameUsage''').fetchall()}
#     for kingdom in kingdoms:
#         where = None
#         if not kingdom:
#             where = f'LOWER("col:kingdom") IS NULL'
#         else:
#             where = f'LOWER("col:kingdom") = \'{kingdom.lower()}\''
        
#         s.append({
#             'level': 0,
#             'label': 'Kingdom',
#             'value': kingdom,
#             'where': [where],
#             'parent': '*',
#         })
    
#     taken = {}
#     explored = 0
    
#     while s:
#         top = s.pop()
            
#         # Generate ID for Node
#         identifier = f"{top['label']}:{top['value']}"
#         if identifier in taken:
#             taken[identifier] += 1
#             identifier += f'-{taken[identifier] + 1}'
#         else:
#             taken[identifier] = 1
#             identifier += f'-1'

#         # Create Node
#         t.create_node(
#             identifier=identifier,
#             tag=top['value'],
#             parent=top['parent'],
#             data={
#                 'label': top['label'],
#                 'value': top['value']
#             }
#         )
        
#         # Find Node Descendants
#         n_level = top['level'] + 1
#         if n_level >= len(levels):
#             explored += 1
#             clear_output(wait=True)
#             print(f'Explored: {explored}')
#             continue
        
#         n_label, n_column = levels[n_level]
#         n_where = ' AND '.join(top['where'])
#         n_values = {row[0] for row in cursor.execute(f"""SELECT DISTINCT "{n_column}" FROM NameUsage WHERE {n_where}""").fetchall()}
    
#         for n_value in n_values:
#             where = None
#             if not n_value:
#                 where = f'LOWER("{n_column}") IS NULL'
#             else:
#                 where = f'LOWER("{n_column}") = \'{n_value.lower()}\''
            
#             s.append({
#                 'level': n_level,
#                 'label': n_label,
#                 'value': n_value,
#                 'where': [*top['where'], where],
#                 'parent': identifier
#             })
    
#     return t

# tree = create_levels(cursor)

In [360]:
# Already Ran
for col in cols:
    conn.execute(f'''
        CREATE INDEX IF NOT EXISTS Index{col[0]}LowerCase ON NameUsage (LOWER("{col[1]}"))
    ''')

In [369]:
data = []

cols = [["kingdom","col:kingdom"],["phylum","col:phylum"],["subphylum","col:subphylum"],["class","col:class"],["subclass","col:subclass"],["order","col:order"],["suborder","col:suborder"],["superfamily","col:superfamily"],["family","col:family"],["subfamily","col:subfamily"],["genus","col:genus"],["subgenus","col:subgenus"],["genericName","col:genericName"],["specificEpithet","col:specificEpithet"],["intraspecificEpithet","col:infraspecificEpithet"],["species","col:species"],["scientificName","col:scientificName"]]
for col in cols:
    labels = set()

    # Regex for (X) Y
    regex = r'^([A-Za-z]+)\s\(([A-Za-z]+)\)$'
    
    for val in df[col[1]].tolist():
        if not val:
            continue

        if col[0] == "Subgenus":
            match = re.match(regex, val.lower())
            labels.add(val.lower() if not match else match.group(2))
        else:
            labels.add(val.lower())
    
    data.append({
        'column': col[1],
        'column_label': col[0],
        'values': labels
    })
    
# Save File
with open(f"Taxa.pickle", "wb") as file:
    pickle.dump(data, file)

In [354]:
data[-1]['column_label']

'ScientificName'

In [355]:
s1 = 'panthera leo'
s2 = 'panthera leo leo'

In [363]:
def is_related(s1, s2):
    buffer = []

    s1 = s1.lower()
    s2 = s2.lower()

    if s1 == s2:
        return True
    
    used_s1 = False
    used_s2 = False
    
    for d in data:
        if used_s1 and used_s2:
            break
        
        if not used_s1 and s1 in d['values']:
            buffer.append((s1, d['column']))
            used_s1 = True
            continue

        if not used_s2 and s2 in d['values']:
            buffer.append((s2, d['column']))
            used_s2 = True
            continue

    print(buffer)
    
    if len(buffer) != 2:
        return False
        
    (t1_val, t1_col), (t2_val, t2_col) = buffer
    
    if t1_col == t2_col:
        return t1_val == t2_val
    
    # q = f'''
    #     SELECT LOWER("{t1_col}")
    #     FROM NameUsage
    #     WHERE LOWER("{t2_col}") = \'{t2_val}\'
    # '''

    # q = f'''
    #     SELECT 1 
    #     WHERE '{t1_val}' IN (
    #         SELECT LOWER("{t1_col}")
    #         FROM NameUsage
    #         WHERE LOWER("{t2_col}") = \'{t2_val}\'
    #     )
    #     LIMIT 1
    # '''

    q = f'''
        SELECT 1
        FROM NameUsage
        WHERE LOWER("{t2_col}") = \'{t2_val}\' AND LOWER("{t1_col}") = \'{t1_val}\'
        LIMIT 1
    '''
    
    output = conn.execute(q)
    return bool(output.fetchone())
    # while batch := output.fetchmany(800):
    #     if t1_val in {row[0] for row in batch}:
    #         return True
    
    # return False

t0 = time.time()
r = is_related(s1, s2)
t1 = time.time()
print(r)
print(f'Time: {t1-t0}s')

[('panthera leo', 'col:species'), ('panthera leo leo', 'col:scientificName')]
True
Time: 0.0012373924255371094s


In [ ]:
data[1]['values']

In [ ]:
t0 = time.time()

s1_col = None
s2_col = None
for d in data:
    if not s1_col and s1 in d['values']:
        s1_col = d['column']

    if not s2_col and s2 in d['values']:
        s2_col = d['column']

if s1_col and s2_col:
    # q = f'''
    #     SELECT 1 
    #     FROM NameUsage 
    #     WHERE LOWER("{s1_col}") = \'{s1}\' AND LOWER("{s2_col}") = \'{s2}\'
    #     LIMIT 1
    # '''

    q = f'''
        SELECT LOWER("{s1_col}")
        FROM NameUsage
        WHERE LOWER("{s2_col}") = \'{s2}\'
    '''
    
    subs = set([row[0] for row in cursor.execute(q).fetchall()])
    if s1 in subs:
        print('Relationship')

t1 = time.time()
print(f'Time: {t1-t0}s')

In [ ]:
print(s1_col)
print(s2_col)

In [ ]:
t0 = time.time()
# q = f'''
#     SELECT "{s1_col}"
#     FROM NameUsage
#     WHERE LOWER("col:phylum") = \'porfirea\'
# '''
# q = f'''
#     SELECT 1 WHERE 'chordata' IN (SELECT LOWER("col:phylum") FROM NameUsage WHERE LOWER("col:kingdom") = 'animalia') LIMIT 1
# '''
# q = f'''
#     SELECT 1 
#     WHERE 'animalia' IN (
#         SELECT LOWER("col:kingdom")
#         FROM NameUsage
#         WHERE LOWER("col:phylum") = 'porifera'
#     )
#     LIMIT 1
# '''
q = '''
    SELECT 1
    FROM NameUsage
    WHERE  LOWER("col:genus") = 'leo' AND LOWER("col:kingdom") = 'animalia'
    LIMIT 1
'''
# cursor.execute(q)
cursor.execute(f'EXPLAIN QUERY PLAN {q}')
t1 = time.time()
# cursor.execute(f'EXPLAIN QUERY PLAN SELECT 1 FROM NameUsage WHERE LOWER("col:kingdom") = ? AND LOWER("col:phylum") = ? LIMIT 1', ('animalia', 'chordata'))
print(cursor.fetchall())
print(t1-t0)

In [358]:
cursor = conn.cursor()
cursor.execute("SELECT name, sql FROM sqlite_master WHERE type='index'")
print(cursor.fetchall())
# print(cursor.execute('SELECT COUNT(DISTINCT LOWER("col:phylum")) FROM NameUsage').fetchone()[0])

[('IndexName', 'CREATE INDEX IndexName ON VernacularName ("col:name")'), ('IndexID', 'CREATE INDEX IndexID ON NameUsage ("col:ID")'), ('IndexTaxonID', 'CREATE INDEX IndexTaxonID ON VernacularName ("col:taxonID")'), ('IndexRank', 'CREATE INDEX IndexRank ON NameUsage ("col:rank")'), ('IndexLanguage', 'CREATE INDEX IndexLanguage ON VernacularName ("col:language")'), ('IndexScientificLowerCase', 'CREATE INDEX IndexScientificLowerCase ON NameUsage (LOWER("col:scientificName"))'), ('IndexScientificNameLowerCase', 'CREATE INDEX IndexScientificNameLowerCase ON NameUsage (LOWER("col:scientificName"))'), ('IndexGenericNameLowerCase', 'CREATE INDEX IndexGenericNameLowerCase ON NameUsage (LOWER("col:genericName"))'), ('IndexGenericNameLowerCaseFirstCharacter', 'CREATE INDEX IndexGenericNameLowerCaseFirstCharacter ON NameUsage (SUBSTR(LOWER("col:genericName"), 1, 1))'), ('IndexSpecificEpithetLowerCase', 'CREATE INDEX IndexSpecificEpithetLowerCase ON NameUsage (LOWER("col:specificEpithet"))'), ('Ind

In [ ]:
df.columns

In [ ]:
df["col:status"].value_counts()

In [ ]:
df["col:rank"].value_counts()

In [ ]:
df[df["col:rank"] == "subgenus"].head(1)

In [ ]:
subgenera = set(df["col:species"].tolist())


In [ ]:
subgenera

In [287]:
i = 0
for subgenus in subgenera:
    if not subgenus:
        continue
    if i >= 50:
        break
    i += 1
    split = subgenus.split()
    if len(split) == 2:
        print(subgenus)

Placusa nairobiana
Aiouea amplexicaulis
Zygaena vogesiaca
Isolona pleurocarpa
Fusicoccum mangiferae
Pachysmopoda abbreviata
Huperzia cleefiana
Lycium lineare
Signiphora falcata
Orthops alpicola
Syntomus antoinei
Prochaetosoma arcticum
Cytheropteron branchium
Asarum caulescens
Chrysopa kiangsuensis
Pycnostachys graminifolia
Ophiosphalma serratum
Diaporthe chamaeropicola
Turritella australis
Philarctus bergrothi
Leptonia modesta
Bioculus belvederi
Euryphorus hemipterus
Polystichum caripense
Spongostylum variabile
Cyrtochloa major
Simulium yamayaense
Conidiobolus undulatus


In [371]:
cursor = conn.cursor()
cursor.executescript('''
    DROP TABLE IF EXISTS MappedName;
    
    CREATE TABLE IF NOT EXISTS MappedName AS
    SELECT n."col:ID" AS ID,
           n."col:scientificName" AS ScientificName,
           v."col:name" AS VernacularName,
           n."col:kingdom" AS Kingdom,
           n."col:phylum" AS Phylum,
           n."col:subphylum" AS SubPhylum,
            n."col:class" AS Class,
            n."col:subclass" AS SubClass,
            n."col:order" AS TaxonOrder,
            n."col:suborder" AS SubOrder,
            n."col:superfamily" AS SuperFamily,
            n."col:family" AS Family,
            n."col:subfamily" AS SubFamily,
            n."col:genus" AS Genus,
            n."col:subgenus" AS SubGenus,
            n."col:genericName" AS GenericName,
            n."col:specificEpithet" AS SpecificEpithet,
            n."col:infraspecificEpithet" AS IntraspecificEpithet,
            n."col:species" AS Species
    FROM NameUsage n
    JOIN VernacularName v ON n."col:ID" = v."col:taxonID"
    WHERE v."col:language" = 'eng';

    CREATE INDEX IF NOT EXISTS IndexID ON MappedName(LOWER("ID"));
    CREATE INDEX IF NOT EXISTS IndexScientificName ON MappedName(LOWER("ScientificName"));
    CREATE INDEX IF NOT EXISTS IndexVernacularName ON MappedName(LOWER("VernacularName"));
    CREATE INDEX IF NOT EXISTS IndexGenericName1 ON MappedName (SUBSTR(LOWER("GenericName"), 1, 1));
    CREATE INDEX IF NOT EXISTS IndexKingdom ON MappedName(LOWER("Kingdom"));
    CREATE INDEX IF NOT EXISTS IndexPhylum ON MappedName(LOWER("Phylum"));
    CREATE INDEX IF NOT EXISTS IndexSubPhylum ON MappedName(LOWER("SubPhylum"));
    CREATE INDEX IF NOT EXISTS IndexClass ON MappedName(LOWER("Class"));
    CREATE INDEX IF NOT EXISTS IndexSubClass ON MappedName(LOWER("SubClass"));
    CREATE INDEX IF NOT EXISTS IndexOrder ON MappedName(LOWER("TaxonOrder"));
    CREATE INDEX IF NOT EXISTS IndexSubOrder ON MappedName(LOWER("SubOrder"));
    CREATE INDEX IF NOT EXISTS IndexSuperFamily ON MappedName(LOWER("SuperFamily"));
    CREATE INDEX IF NOT EXISTS IndexFamily ON MappedName(LOWER("Family"));
    CREATE INDEX IF NOT EXISTS IndexSubFamily ON MappedName(LOWER("SubFamily"));
    CREATE INDEX IF NOT EXISTS IndexGenus ON MappedName(LOWER("Genus"));
    CREATE INDEX IF NOT EXISTS IndexSubGenus ON MappedName(LOWER("SubGenus"));
    CREATE INDEX IF NOT EXISTS IndexGenericNames ON MappedName(LOWER("GenericName"));
    CREATE INDEX IF NOT EXISTS IndexSpecificEpithet ON MappedName(LOWER("SpecificEpithet"));
    CREATE INDEX IF NOT EXISTS IndexIntraspecificEpithet ON MappedName(LOWER("IntraspecificEpithet"));
    CREATE INDEX IF NOT EXISTS IndexSpecies ON MappedName(LOWER("Species"));
''')
cursor.close()

In [372]:
df.columns

Index(['col:ID', 'col:status', 'col:scientificName', 'col:rank',
       'col:originalSpelling', 'col:uninomial', 'col:genericName',
       'col:infragenericEpithet', 'col:specificEpithet',
       'col:infraspecificEpithet', 'col:code', 'col:nameStatus',
       'col:environment', 'col:species', 'col:subgenus', 'col:genus',
       'col:subfamily', 'col:family', 'col:superfamily', 'col:suborder',
       'col:order', 'col:subclass', 'col:class', 'col:subphylum', 'col:phylum',
       'col:kingdom'],
      dtype='object')

In [374]:
df[df["col:species"].isna()]

,col:ID,col:status,col:scientificName,col:rank,col:originalSpelling,col:uninomial,col:genericName,col:infragenericEpithet,col:specificEpithet,col:infraspecificEpithet,...,col:subfamily,col:family,col:superfamily,col:suborder,col:order,col:subclass,col:class,col:subphylum,col:phylum,col:kingdom
0,8L3XD,synonym,Bembidion littorale,species,None,None,Bembidion,None,littorale,None,...,None,None,None,None,None,None,None,None,None,None
1,CG9YH,accepted,Potamocypris granulosa,species,None,None,Potamocypris,None,granulosa,None,...,Cypridopsinae,Cyprididae,Cypridoidea,Cypridocopina,Podocopida,Podocopa,Ostracoda,Crustacea,Arthropoda,Animalia
2,ZFNQ,accepted,Cribrilina punctata,species,None,None,Cribrilina,None,punctata,None,...,None,Cribrilinidae,Cribrilinoidea,Flustrina,Cheilostomatida,None,Gymnolaemata,None,Bryozoa,Animalia
3,3B3PZ,accepted,Eriochloa subulifera,species,None,None,Eriochloa,None,subulifera,None,...,None,Poaceae,None,None,Poales,None,Liliopsida,None,Tracheophyta,Plantae
4,8CSJ3,synonym,Psectrocladius sokolovae,species,None,None,Psectrocladius,None,sokolovae,None,...,None,None,None,None,None,None,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7917924,N3PTV,accepted,Dichogaster notabilis,species,None,None,Dichogaster,None,notabilis,None,...,None,Benhamiidae,None,Lumbricina,Crassiclitellata,Oligochaeta,Clitellata,None,Annelida,Animalia
7917925,N3PW4,accepted,Dichogaster papillosa,species,None,None,Dichogaster,None,papillosa,None,...,None,Benhamiidae,None,Lumbricina,Crassiclitellata,Oligochaeta,Clitellata,None,Annelida,Animalia
7917926,N3PWN,accepted,Dichogaster penigera,species,None,None,Dichogaster,None,penigera,None,...,None,Benhamiidae,None,Lumbricina,Crassiclitellata,Oligochaeta,Clitellata,None,Annelida,Animalia
7917927,N3PXJ,accepted,Dichogaster proandra,species,None,None,Dichogaster,None,proandra,None,...,None,Benhamiidae,None,Lumbricina,Crassiclitellata,Oligochaeta,Clitellata,None,Annelida,Animalia


In [375]:
len(df)

7917929

In [2]:
import ahocorasick

In [381]:
!pip install pyahocorasick

  Using cached pyahocorasick-2.3.0-cp310-cp310-win_amd64.whl.metadata (14 kB)
Using cached pyahocorasick-2.3.0-cp310-cp310-win_amd64.whl (35 kB)


DEPRECATION: textract 1.6.5 has a non-standard dependency specifier extract-msg<=0.29.*. pip 24.1 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of textract or contact the author to suggest that they release a version with a conforming dependency specifiers. Discussion can be found at https://github.com/pypa/pip/issues/12063

[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [383]:
len(data)

17

In [4]:
def load_data(path: str, load):
    data = None
    if os.path.exists(path):
        with open(path, "rb") as file:
            data = pickle.load(file)
    else:
        data = load()
        with open(path, "wb") as file:
            pickle.dump(data, file)
    return data

In [10]:
def load_scientific_names():
    df = pd.read_sql(f"""
        SELECT *
        FROM NameUsage
    """, conn)

    data = []
    cols = [["kingdom","col:kingdom"],["phylum","col:phylum"],["subphylum","col:subphylum"],["class","col:class"],["subclass","col:subclass"],["order","col:order"],["suborder","col:suborder"],["superfamily","col:superfamily"],["family","col:family"],["subfamily","col:subfamily"],["genus","col:genus"],["subgenus","col:subgenus"],["genericName","col:genericName"],["specificEpithet","col:specificEpithet"],["intraspecificEpithet","col:infraspecificEpithet"],["species","col:species"],["scientificName","col:scientificName"]]
    
    for col in cols:
        values = set()
        regex = r'^([A-Za-z]+)\s\(([A-Za-z]+)\)$'
        
        for value in df[col[1]].tolist():
            if not value:
                continue

            if col[0] != "subgenus":
                values.add(value.lower())
                continue

            # The sub-genus is placed inside '(...)',
            # so we need to capture it.
            match = re.match(regex, value.lower())
            values.add(value.lower() if not match else match.group(2))
        
        data.append({
            'column_label': col[0],
            'column': col[1],
            'values': values
        })

    del df
    return data

names = load_data("ScientificNames.pickle", load_scientific_names)
print(f"# of Names: {len(names)}")

# of Names: 17


In [33]:
import ahocorasick

t0 = time.time()

A = ahocorasick.Automaton()

# i = 0
# for d in names:
#     for v in d['values']:
#         A.add_word(v, (d['column_label'], v))
A.add_word('word', 'anything')

A.make_automaton()

print(A.get('word'))

text = 'This is a word, but you are a noun'
for r_i, data in A.iter(text):
    print(data)
    # print(r_i, label, key, text[r_i - len(key) + 1:r_i+1])

t1 = time.time()

anything
anything


In [17]:
print(f'{t1-t0}s')

81.71150803565979s


In [26]:
text = '''During a survey of the Amazon basin, researchers documented several species including Panthera onca (jaguar), Ara macao (scarlet macaw), and Eunectes murinus (green anaconda) along the riverbanks. In the canopy above, Alouatta seniculus (red howler monkey) and Bradypus tridactylus (pale-throated sloth) were observed feeding on Ficus insipida leaves. Aquatic sampling revealed Arapaima gigas, one of the world's largest freshwater fish, alongside Electrophorus electricus (electric eel). Entomologists also recorded Morpho menelaus (blue morpho butterfly) and the leafcutter ant Atta cephalotes constructing their colonies beneath Cedrela odorata trees."'''
for end_idx, (idx, key) in A.iter(text.lower()):
    l = text[end_idx - len(key)]
    r = text[end_idx + 1]

    # print(l, key, r)
    if (l.isspace() or l in string.punctuation) and (r.isspace() or r in string.punctuation):
        print(f"Found {key} at position {end_idx}")        

Found a at position 7
Found the at position 21
Found amazon at position 28
Found species at position 74
Found panthera at position 93
Found panthera onca at position 98
Found onca at position 98
Found jaguar at position 106
Found ara at position 112
Found ara macao at position 118
Found macao at position 118
Found eunectes at position 148
Found eunectes murinus at position 156
Found murinus at position 156
Found the at position 183
Found in at position 198
Found the at position 202
Found canopy at position 209
Found alouatta at position 225
Found alouatta seniculus at position 235
Found seniculus at position 235
Found bradypus at position 268
Found bradypus tridactylus at position 280
Found tridactylus at position 280
Found on at position 327
Found ficus at position 333
Found ficus insipida at position 342
Found insipida at position 342
Found leaves at position 349
Found aquatic at position 358
Found arapaima at position 385
Found arapaima gigas at position 391
Found gigas at position 